In [1]:
import pandas as pd

In [3]:
df_gmv = pd.read_csv("./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv", encoding='utf-8-sig')


### Thống kê sdt của gmv và lead

In [5]:
# Hàm đếm độ dài số điện thoại
def phone_length_stats(df, phone_col):
    # Chuyển về string
    phones = (
        df[phone_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Chỉ giữ lại số
    phones = phones.str.replace(r"\D", "", regex=True)

    # Tính độ dài
    lengths = phones.str.len()

    # Đếm tần suất
    stats = (
        lengths.value_counts()
        .sort_index()
        .rename_axis("Phone Length")
        .reset_index(name="Count")
    )

    return stats

# GMV
print("===== GMV =====")
print(phone_length_stats(df_gmv, "Phone"))

# Leads
print("\n===== LEADS =====")
print(phone_length_stats(df_leads, "Số điện thoại"))

===== GMV =====
   Phone Length  Count
0            11    345
1            12    101
2            13     17

===== LEADS =====
   Phone Length  Count
0             0    229
1             4      1
2            10      9
3            11  14372
4            12   1560
5            13    160
6            14      3
7            15      2


In [8]:
def phone_length_examples(df, phone_col, n=5):
    temp = pd.DataFrame({
        "Raw": df[phone_col].fillna("").astype(str)
    })

    # Chỉ dùng để tính độ dài
    temp["Clean"] = (
        temp["Raw"]
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    temp["Length"] = temp["Clean"].str.len()

    for length in sorted(temp["Length"].unique()):
        print(f"\n===== Length = {length} =====")

        samples = (
            temp[temp["Length"] == length]
            [["Raw", "Clean"]]
            .drop_duplicates()
            .head(n)
        )

        print(samples.to_string(index=False))
        
print("===== GMV =====")
phone_length_examples(df_gmv, "Phone")

print("\n===== LEADS =====")
phone_length_examples(df_leads, "Số điện thoại")

===== GMV =====

===== Length = 11 =====
         Raw       Clean
84-908224672 84908224672
84-909517679 84909517679
84-964678701 84964678701
84-389970625 84389970625
84-938593961 84938593961

===== Length = 12 =====
          Raw        Clean
420-734602066 420734602066
81-7031882708 817031882708
81-8035704573 818035704573
81-7085394685 817085394685
81-8051986529 818051986529

===== Length = 13 =====
           Raw         Clean
49-15750418390 4915750418390
49-17644461422 4917644461422
49-15205315239 4915205315239
49-17641893888 4917641893888
49-17687465203 4917687465203

===== LEADS =====

===== Length = 0 =====
    Raw Clean
             
QR ZALO      
      -      
  MÃ QR      
QR Zalo      

===== Length = 4 =====
  Raw Clean
16:33  1633

===== Length = 10 =====
        Raw      Clean
82-66831308 8266831308
84-35449883 8435449883
84-77705527 8477705527
82-66248699 8266248699
84-56562448 8456562448

===== Length = 11 =====
         Raw       Clean
84-919249237 84919249237
84-9654685

### Thống kê uid của gmv và leads

In [6]:
def uid_length_stats(df, uid_col):
    # Chuẩn hóa UID
    uids = (
        df[uid_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()      # hoặc .str.lower()
    )

    # Tính độ dài
    lengths = uids.str.len()

    # Thống kê
    stats = (
        lengths.value_counts()
        .sort_index()
        .rename_axis("UID Length")
        .reset_index(name="Count")
    )

    return stats


# GMV
print("===== GMV UID =====")
print(uid_length_stats(df_gmv, "UID"))

# Leads
print("\n===== LEADS UID =====")
print(uid_length_stats(df_leads, "UID"))

===== GMV UID =====
   UID Length  Count
0          10    463

===== LEADS UID =====
   UID Length  Count
0           0    761
1           1      1
2           2      1
3           9      4
4          10   8759
5          11      5
6          12   6805


In [7]:
def uid_length_examples(df, uid_col, n=5):
    uids = (
        df[uid_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    temp = pd.DataFrame({
        "UID": uids,
        "Length": uids.str.len()
    })

    for length in sorted(temp["Length"].unique()):
        print(f"\n===== Length = {length} =====")
        print(
            temp[temp["Length"] == length]["UID"]
            .drop_duplicates()
            .head(n)
            .tolist()
        )

print("GMV")
uid_length_examples(df_gmv, "UID")

print("\nLEADS")
uid_length_examples(df_leads, "UID")

GMV

===== Length = 10 =====
['3311001921', '3179205818', '3311304274', '3310902627', '3309271098']

LEADS

===== Length = 0 =====
['']

===== Length = 1 =====
['`']

===== Length = 2 =====
['L8']

===== Length = 9 =====
['309845189', '310072108', '307486256', '307272816']

===== Length = 10 =====
['3309599159', '3309599173', '3309599181', '3309622104', '3288358317']

===== Length = 11 =====
['310991875.0', '305383787.0', '158072687.0', '305580541.0', '305825571.0']

===== Length = 12 =====
['3310947243.0', '3310952712.0', '3293618952.0', '3310947249.0', '3310952706.0']


### Clean sdt và phone của cả 2


In [9]:
import pandas as pd
import numpy as np

# ==========================
# CONFIG
# ==========================

GMV_FILE = "./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv"
LEADS_FILE = "./cleaned_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv"

GMV_PHONE_COL = "Phone"
GMV_UID_COL = "UID"

LEADS_PHONE_COL = "Số điện thoại"
LEADS_UID_COL = "UID"


# ==========================
# CLEAN UID
# ==========================

def clean_uid(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

    s = s.replace("", np.nan)

    return s


# ==========================
# CLEAN PHONE
# ==========================

def clean_phone(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    # quá ngắn hoặc quá dài -> invalid
    s = s.where(s.str.len().between(9, 13), np.nan)

    return s


# ==========================
# LOAD
# ==========================

df_gmv = pd.read_csv(GMV_FILE)
df_leads = pd.read_csv(LEADS_FILE)


# ==========================
# CLEAN
# ==========================

df_gmv["UID_clean"] = clean_uid(df_gmv[GMV_UID_COL])
df_gmv["Phone_clean"] = clean_phone(df_gmv[GMV_PHONE_COL])

df_leads["UID_clean"] = clean_uid(df_leads[LEADS_UID_COL])
df_leads["Phone_clean"] = clean_phone(df_leads[LEADS_PHONE_COL])


# ==========================
# EXPORT
# ==========================

df_gmv.to_csv(
    "gmv_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_leads.to_csv(
    "leads_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!


### Check douplicate uid trong gmv


In [79]:
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding='utf-8-sig')


phone_stats = (
    df_gmv.groupby("Phone_clean")
    .size()
    .reset_index(name="Count")
)

phone_dup = phone_stats[phone_stats["Count"] > 1]

print(f"Phone unique: {phone_stats.shape[0]:,}")
print(f"Duplicate phones: {phone_dup.shape[0]:,}")
print(f"Rows involved: {phone_dup['Count'].sum():,}")
print(phone_dup.sort_values("Count", ascending=False).head(20))

Phone unique: 457
Duplicate phones: 6
Rows involved: 12
      Phone_clean  Count
83    84865800589      2
202   84943111987      2
204   84944022155      2
355  491631303303      2
381  818033281246      2
428  821040844054      2


In [80]:
uid_stats = (
    df_gmv.groupby("UID_clean")
    .size()
    .reset_index(name="Count")
)

uid_dup = uid_stats[uid_stats["Count"] > 1]

print(f"UID unique: {uid_stats.shape[0]:,}")
print(f"Duplicate UIDs: {uid_dup.shape[0]:,}")
print(f"Rows involved: {uid_dup['Count'].sum():,}")
print(uid_dup.sort_values("Count", ascending=False).head(20))

UID unique: 454
Duplicate UIDs: 9
Rows involved: 18
      UID_clean  Count
98   3309549687      2
110  3310023176      2
139  3310871114      2
318  3311684368      2
326  3311716270      2
334  3311738141      2
359  3311861764      2
446  3312234999      2
451  3312285639      2


In [81]:
pair_stats = (
    df_gmv.groupby(["Phone_clean", "UID_clean"])
    .size()
    .reset_index(name="Count")
)

pair_dup = pair_stats[pair_stats["Count"] > 1]

print(f"Unique pairs: {pair_stats.shape[0]:,}")
print(f"Duplicate pairs: {pair_dup.shape[0]:,}")
print(f"Rows involved: {pair_dup['Count'].sum():,}")
print(pair_dup.sort_values("Count", ascending=False).head(20))

Unique pairs: 457
Duplicate pairs: 6
Rows involved: 12
      Phone_clean   UID_clean  Count
83    84865800589  3309549687      2
202   84943111987  3312285639      2
204   84944022155  3311861764      2
355  491631303303  3310871114      2
381  818033281246  3311738141      2
428  821040844054  3311716270      2


In [82]:
pair_stats = (
    df_gmv.groupby(["Phone_clean", "UID_clean"])
    .size()
    .reset_index(name="Count")
)

# Các cặp xuất hiện đúng 1 lần
pair_unique = pair_stats[pair_stats["Count"] == 1]

print(f"Total GMV rows           : {len(df_gmv):,}")
print(f"Distinct Phone+UID pairs : {len(pair_stats):,}")
print(f"Unique Phone+UID pairs   : {len(pair_unique):,}")
print(f"Unique rate              : {len(pair_unique) / len(pair_stats):.2%}")

Total GMV rows           : 463
Distinct Phone+UID pairs : 457
Unique Phone+UID pairs   : 451
Unique rate              : 98.69%


### Tìm cách match

In [83]:
df_gmv = pd.read_csv(
    "./cleaned_data/gmv_clean.csv",
    dtype={
        "Phone_clean": str,
        "UID_clean": str
    },
    encoding="utf-8-sig"
)

df_leads = pd.read_csv(
    "./cleaned_data/leads_clean.csv",
    dtype={
        "Phone_clean": str,
        "UID_clean": str
    },
    encoding="utf-8-sig"
)

In [92]:
# Extract unique combinations of Phone and UID into a new DataFrame
unique_gmv_keys_df = df_gmv[["Phone_clean", "UID_clean"]].drop_duplicates()

# Reset index to make it clean
unique_gmv_keys_df = unique_gmv_keys_df.reset_index(drop=True)

print(f"Tổng số cặp key duy nhất trong GMV: {len(unique_gmv_keys_df)}")
print(unique_gmv_keys_df.head())

Tổng số cặp key duy nhất trong GMV: 457
   Phone_clean   UID_clean
0  84908224672  3311001921
1  84909517679  3179205818
2  84964678701  3311304274
3  84389970625  3310902627
4  84938593961  3309271098


In [93]:
# Lọc ra các cặp Phone và UID không trùng lặp từ Leads
unique_leads_keys_df = df_leads[["Phone_clean", "UID_clean"]].drop_duplicates()

# Reset lại index cho gọn gàng
unique_leads_keys_df = unique_leads_keys_df.reset_index(drop=True)

# Loại bỏ những dòng mà cả Phone và UID đều rỗng (nếu có)
unique_leads_keys_df = unique_leads_keys_df[
    (unique_leads_keys_df["Phone_clean"] != "") | 
    (unique_leads_keys_df["UID_clean"] != "")
]

print(f"Tổng số cặp key duy nhất trong Leads: {len(unique_leads_keys_df)}")
print(unique_leads_keys_df.head())

Tổng số cặp key duy nhất trong Leads: 15829
   Phone_clean   UID_clean
0          NaN         NaN
1  84919249237  3310947243
2  84965468525  3310952712
3  84985951781  3293618952
4  84972856556  3310947249


In [96]:
# Tìm tập giao (intersection) giữa 2 bảng dựa trên cả 2 cột
chung_ca_hai = pd.merge(unique_gmv_keys_df, unique_leads_keys_df, on=["Phone_clean", "UID_clean"], how="inner")

print(f"Số lượng khách hàng khớp uid + phone: {len(chung_ca_hai)}")

Số lượng khách hàng khớp uid + phone: 259


In [98]:
# 1. Tạo Set cho GMV (Tự động lọc unique)
set_gmv = set(zip(df_gmv["Phone_clean"], df_gmv["UID_clean"]))

# 2. Tạo Set cho Leads (Dù trùng bao nhiêu lần, ném vào đây cũng tự động gộp thành 1)
set_leads = set(zip(df_leads["Phone_clean"], df_leads["UID_clean"]))

# 3. Loại bỏ rác: Xóa cặp rỗng (khách không có cả phone lẫn UID) để tránh match sai
set_gmv.discard(("", ""))
set_leads.discard(("", ""))

# 4. Tìm Giao điểm (Intersection): Những cặp vừa có trong GMV, vừa có trong Leads
matched_keys = set_gmv.intersection(set_leads)

print(f"Số cặp unique trong GMV: {len(set_gmv)}")
print(f"Số cặp unique trong Leads: {len(set_leads)}")
print("-" * 40)
print(f"🚀 TỔNG SỐ CẶP MATCH THÀNH CÔNG: {len(matched_keys)}")

# Nếu muốn xem thử 5 cặp match được:
print("\nVí dụ 5 cặp match được (Phone, UID):")
print(list(matched_keys)[:5])

Số cặp unique trong GMV: 457
Số cặp unique trong Leads: 15829
----------------------------------------
🚀 TỔNG SỐ CẶP MATCH THÀNH CÔNG: 259

Ví dụ 5 cặp match được (Phone, UID):
[('84982135774', '3311850607'), ('84986694373', '3311939526'), ('84908655015', '3311572089'), ('818042129159', '3311534573'), ('84937973897', '3302383080')]


In [ ]:
# Assuming matched_keys is the set of tuples from the previous step:
# matched_keys = set_gmv.intersection(set_leads)

# 1. Create a boolean mask to identify which rows in GMV match the keys
is_already_matched = df_gmv.set_index(["Phone_clean", "UID_clean"]).index.isin(matched_keys)

# 2. Filter out the matched ones using the ~ (NOT) operator
unmatched_gmv = df_gmv[~is_already_matched]


matched_gmv = df_gmv[is_already_matched]

# Print the results to verify the split
print(f"Total original GMV rows: {len(df_gmv)}")
print(f"Matched GMV rows (Removed): {len(matched_gmv)}")
print(f"Remaining Unmatched GMV rows: {len(unmatched_gmv)}")

Total original GMV rows: 463
Matched GMV rows (Removed): 261
Remaining Unmatched GMV rows: 202


In [100]:
print(unmatched_gmv)

      bank day bank time       Gateway           User Name          Phone  \
0     2026/5/2     18:28       VP Bank    C Thanh - Bé Hân   84-908224672   
1    2026/5/11       NaN           NaN  Chị Nga - Bé Khang   84-909517679   
3    2026/5/14     09:30      Agribank    Chị Huệ - Bé Thi   84-389970625   
6    2026/5/20       NaN           NaN  C Khuê - Huy Hoàng   84-978827804   
9    2026/5/24     20:21       MB Bank    C Dung - Bé Diệp   84-938222653   
..         ...       ...           ...                 ...            ...   
452  2026/5/31     20:58       Shinhan           Đăng Khoa   84-368776525   
458  2026/5/31     21:40  VCB Digibank                  Bé  81-7035351368   
460  2026/5/31         x          VIMO                 Kem   84-969663003   
461  2026/5/31     23:38             x           Thiên Kim   84-943111987   
462  2026/5/31     23:38        MBBank                 Mon   84-943111987   

            UID                              Package Fixed/ non-fixed  \
0 